In [3]:
import os
import pandas as pd

# Define the base path
base_path = r'c:\Files\Degree\FYP\FYP_ver5\Results\10seeds_cv5_runs'

# Define the subdirectories to process
target_dirs = [
    '1_Potential_Models',
    '2_Test_Add_LayerNorm',
    '3_Test_AlternativePreprocessing'
]

# Define platforms
platforms = ['Reddit', 'Twitter']

def merge_histories(f5_path, l5_path, output_path):
    try:
        # Read all sheets from both files
        # sheet_name=None reads all sheets into a dictionary
        f5_sheets = pd.read_excel(f5_path, sheet_name=None)
        l5_sheets = pd.read_excel(l5_path, sheet_name=None)
        
        # Combine the dictionaries
        combined_sheets = {**f5_sheets, **l5_sheets}
        
        # Write to the output file
        with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
            for sheet_name, df in combined_sheets.items():
                df.to_excel(writer, sheet_name=sheet_name, index=False)
                
        print(f"Successfully merged to: {output_path}")
        return True
    except Exception as e:
        print(f"Error merging {f5_path} and {l5_path}: {str(e)}")
        return False

# Main loop
for target_dir in target_dirs:
    for platform in platforms:
        current_path = os.path.join(base_path, target_dir, platform)
        
        if not os.path.exists(current_path):
            print(f"Directory not found: {current_path}")
            continue
            
        print(f"Processing: {current_path}")
        
        # List all directories in the current path
        try:
            subfolders = [f for f in os.listdir(current_path) if os.path.isdir(os.path.join(current_path, f))]
        except Exception as e:
            print(f"Error listing directories in {current_path}: {e}")
            continue

        # Find F5 folders
        f5_folders = [f for f in subfolders if f.startswith('F5S')]
        
        for f5_folder in f5_folders:
            # Extract the suffix (everything after F5S)
            suffix = f5_folder[3:]
            
            # Construct expected L5 and 10S folder names
            l5_folder = f"L5S{suffix}"
            ten_s_folder = f"10S{suffix}"
            
            # Check if L5 and 10S folders exist
            if l5_folder in subfolders and ten_s_folder in subfolders:
                # Construct file paths
                f5_file_name = f"{f5_folder}_training_history.xlsx"
                l5_file_name = f"{l5_folder}_training_history.xlsx"
                ten_s_file_name = f"{ten_s_folder}_training_history.xlsx"
                
                f5_file_path = os.path.join(current_path, f5_folder, f5_file_name)
                l5_file_path = os.path.join(current_path, l5_folder, l5_file_name)
                output_dir_path = os.path.join(current_path, ten_s_folder)
                output_path = os.path.join(current_path, ten_s_folder, ten_s_file_name)
                
                # Check if source files exist
                if os.path.exists(f5_file_path) and os.path.exists(l5_file_path) and os.path.exists(output_dir_path):
                    print(f"Merging for {suffix}...")
                    merge_histories(f5_file_path, l5_file_path, output_path)
                else:
                    if not os.path.exists(f5_file_path):
                        print(f"Missing F5 file: {f5_file_path}")
                    if not os.path.exists(l5_file_path):
                        print(f"Missing L5 file: {l5_file_path}")
                    if not os.path.exists(output_dir_path):
                        print(f"Missing output directory: {output_dir_path}")

Processing: c:\Files\Degree\FYP\FYP_ver5\Results\10seeds_cv5_runs\1_Potential_Models\Reddit
Merging for R_tcn_model_fine_tune_mb_pool...
Successfully merged to: c:\Files\Degree\FYP\FYP_ver5\Results\10seeds_cv5_runs\1_Potential_Models\Reddit\10SR_tcn_model_fine_tune_mb_pool\10SR_tcn_model_fine_tune_mb_pool_training_history.xlsx
Merging for R_tcn_model_fine_tune_mb_seqOut...
Successfully merged to: c:\Files\Degree\FYP\FYP_ver5\Results\10seeds_cv5_runs\1_Potential_Models\Reddit\10SR_tcn_model_fine_tune_mb_seqOut\10SR_tcn_model_fine_tune_mb_seqOut_training_history.xlsx
Processing: c:\Files\Degree\FYP\FYP_ver5\Results\10seeds_cv5_runs\1_Potential_Models\Twitter
Merging for T_tcn_model_fine_tune_mb_pool...
Successfully merged to: c:\Files\Degree\FYP\FYP_ver5\Results\10seeds_cv5_runs\1_Potential_Models\Twitter\10ST_tcn_model_fine_tune_mb_pool\10ST_tcn_model_fine_tune_mb_pool_training_history.xlsx
Merging for T_tcn_model_fine_tune_mb_seqOut...
Successfully merged to: c:\Files\Degree\FYP\FYP_ve

In [4]:
# Logic for 4_ValidateWithOtherDataset
import os
import pandas as pd
base_path = r'c:\Files\Degree\FYP\FYP_ver5\Results\10seeds_cv5_runs'

def merge_multiple_histories(source_paths, output_path):
    try:
        combined_sheets = {}
        files_found = 0
        for path in source_paths:
            if os.path.exists(path):
                print(f"Reading {path}...")
                sheets = pd.read_excel(path, sheet_name=None)
                combined_sheets.update(sheets)
                files_found += 1
            else:
                print(f"Warning: File not found {path}")
        
        if files_found == 0:
            print("No files found to merge.")
            return False

        # Write to the output file
        with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
            for sheet_name, df in combined_sheets.items():
                df.to_excel(writer, sheet_name=sheet_name, index=False)
                
        print(f"Successfully merged {files_found} files to: {output_path}")
        return True
    except Exception as e:
        print(f"Error merging to {output_path}: {str(e)}")
        return False

validate_dir = '4_ValidateWithOtherDataset'
# Mapping from source code (in 1-2S{code}) to target code (in 10S{target_code})
validate_platforms = {
    'RedditSNS': {'code': 'Z', 'target_code': 'Rsns'},
    'Twitter2': {'code': 'X', 'target_code': 'T2'}
}

# The prefixes for the 5 parts
prefixes = ['1-2S', '3-4S', '5-6S', '7-8S', '9-10S']

print(f"Starting validation merge process for platforms: {list(validate_platforms.keys())}")

for platform, codes in validate_platforms.items():
    current_path = os.path.join(base_path, validate_dir, platform)
    
    if not os.path.exists(current_path):
        print(f"Directory not found: {current_path}")
        continue
        
    print(f"\nProcessing {validate_dir}/{platform}...")
    
    try:
        subfolders = [f for f in os.listdir(current_path) if os.path.isdir(os.path.join(current_path, f))]
    except Exception as e:
        print(f"Error listing directories in {current_path}: {e}")
        continue
        
    # Find model types by looking for 1-2S{code} folders
    start_prefix = f"1-2S{codes['code']}"
    model_folders = [f for f in subfolders if f.startswith(start_prefix)]
    
    print(f"Found {len(model_folders)} model folders matching prefix '{start_prefix}'")

    for model_folder in model_folders:
        # Extract the suffix (the part after 1-2SZ or 1-2SX)
        # e.g. 1-2SZ_tcn_model_fine_tune_mb_pool -> _tcn_model_fine_tune_mb_pool
        suffix = model_folder[len(start_prefix):]
        
        print(f"  Found model suffix: {suffix}")
        
        # Construct the list of 5 source files
        source_files = []
        for prefix in prefixes:
            folder_name = f"{prefix}{codes['code']}{suffix}"
            file_name = f"{folder_name}_training_history.xlsx"
            file_path = os.path.join(current_path, folder_name, file_name)
            source_files.append(file_path)
            
        # Construct the target path
        # Target folder should be 10S{target_code}{suffix}
        target_folder_name = f"10S{codes['target_code']}{suffix}"
        target_file_name = f"{target_folder_name}_training_history.xlsx"
        target_path = os.path.join(current_path, target_folder_name, target_file_name)
        
        # Check if target folder exists (we assume we are merging INTO an existing structure)
        if os.path.exists(os.path.join(current_path, target_folder_name)):
            print(f"  Merging into {target_folder_name}...")
            merge_multiple_histories(source_files, target_path)
        else:
            print(f"  Target folder not found: {target_folder_name} (Expected for {suffix})")

Starting validation merge process for platforms: ['RedditSNS', 'Twitter2']

Processing 4_ValidateWithOtherDataset/RedditSNS...
Found 1 model folders matching prefix '1-2SZ'
  Found model suffix: _tcn_model_fine_tune_mb_pool
  Merging into 10SRsns_tcn_model_fine_tune_mb_pool...
Reading c:\Files\Degree\FYP\FYP_ver5\Results\10seeds_cv5_runs\4_ValidateWithOtherDataset\RedditSNS\1-2SZ_tcn_model_fine_tune_mb_pool\1-2SZ_tcn_model_fine_tune_mb_pool_training_history.xlsx...
Reading c:\Files\Degree\FYP\FYP_ver5\Results\10seeds_cv5_runs\4_ValidateWithOtherDataset\RedditSNS\3-4SZ_tcn_model_fine_tune_mb_pool\3-4SZ_tcn_model_fine_tune_mb_pool_training_history.xlsx...
Reading c:\Files\Degree\FYP\FYP_ver5\Results\10seeds_cv5_runs\4_ValidateWithOtherDataset\RedditSNS\5-6SZ_tcn_model_fine_tune_mb_pool\5-6SZ_tcn_model_fine_tune_mb_pool_training_history.xlsx...
Reading c:\Files\Degree\FYP\FYP_ver5\Results\10seeds_cv5_runs\4_ValidateWithOtherDataset\RedditSNS\7-8SZ_tcn_model_fine_tune_mb_pool\7-8SZ_tcn_mod

In [6]:
import os
import pandas as pd

# Define the base path
base_path = r'c:\Files\Degree\FYP\FYP_ver5\Results\10seeds_cv5_runs'

# Define the subdirectories to process
target_dirs = [
    '4_ValidateWithOtherDataset'
]

# Define platforms
platforms = ['RedditSNS', 'Twitter2']
def merge_histories(f5_path, l5_path, output_path):
    try:
        # Read all sheets from both files
        # sheet_name=None reads all sheets into a dictionary
        f5_sheets = pd.read_excel(f5_path, sheet_name=None)
        l5_sheets = pd.read_excel(l5_path, sheet_name=None)
        
        # Combine the dictionaries
        combined_sheets = {**f5_sheets, **l5_sheets}
        
        # Write to the output file
        with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
            for sheet_name, df in combined_sheets.items():
                df.to_excel(writer, sheet_name=sheet_name, index=False)
                
        print(f"Successfully merged to: {output_path}")
        return True
    except Exception as e:
        print(f"Error merging {f5_path} and {l5_path}: {str(e)}")
        return False

# Main loop
for target_dir in target_dirs:
    for platform in platforms:
        current_path = os.path.join(base_path, target_dir, platform)
        
        if not os.path.exists(current_path):
            print(f"Directory not found: {current_path}")
            continue
            
        print(f"Processing: {current_path}")
        
        # List all directories in the current path
        try:
            subfolders = [f for f in os.listdir(current_path) if os.path.isdir(os.path.join(current_path, f))]
        except Exception as e:
            print(f"Error listing directories in {current_path}: {e}")
            continue

        # Find F5 folders
        f5_folders = [f for f in subfolders if f.startswith('F5S')]
        
        for f5_folder in f5_folders:
            # Extract the suffix (everything after F5S)
            suffix = f5_folder[3:]
            
            # Construct expected L5 and 10S folder names
            l5_folder = f"L5S{suffix}"
            ten_s_folder = f"10S{suffix}"
            
            # Check if L5 and 10S folders exist
            if l5_folder in subfolders and ten_s_folder in subfolders:
                # Construct file paths
                f5_file_name = f"{f5_folder}_training_history.xlsx"
                l5_file_name = f"{l5_folder}_training_history.xlsx"
                ten_s_file_name = f"{ten_s_folder}_training_history.xlsx"
                
                f5_file_path = os.path.join(current_path, f5_folder, f5_file_name)
                l5_file_path = os.path.join(current_path, l5_folder, l5_file_name)
                output_dir_path = os.path.join(current_path, ten_s_folder)
                output_path = os.path.join(current_path, ten_s_folder, ten_s_file_name)
                
                # Check if source files exist
                if os.path.exists(f5_file_path) and os.path.exists(l5_file_path) and os.path.exists(output_dir_path):
                    print(f"Merging for {suffix}...")
                    merge_histories(f5_file_path, l5_file_path, output_path)
                else:
                    if not os.path.exists(f5_file_path):
                        print(f"Missing F5 file: {f5_file_path}")
                    if not os.path.exists(l5_file_path):
                        print(f"Missing L5 file: {l5_file_path}")
                    if not os.path.exists(output_dir_path):
                        print(f"Missing output directory: {output_dir_path}")

Processing: c:\Files\Degree\FYP\FYP_ver5\Results\10seeds_cv5_runs\4_ValidateWithOtherDataset\RedditSNS
Processing: c:\Files\Degree\FYP\FYP_ver5\Results\10seeds_cv5_runs\4_ValidateWithOtherDataset\Twitter2
Merging for T2_baseline_algo...
Successfully merged to: c:\Files\Degree\FYP\FYP_ver5\Results\10seeds_cv5_runs\4_ValidateWithOtherDataset\Twitter2\10ST2_baseline_algo\10ST2_baseline_algo_training_history.xlsx


In [4]:
import os
import pandas as pd

# Define the base path
base_path = r'c:\Files\Degree\FYP\FYP_ver5\Results'

# Define the subdirectories to process
target_dirs = [
    '30seeds_results'
]

# Define platforms
# platforms = ['RedditSNS', 'Twitter2']
def merge_histories(f5_path, l5_path, output_path):
    try:
        # Read all sheets from both files
        # sheet_name=None reads all sheets into a dictionary
        f5_sheets = pd.read_excel(f5_path, sheet_name=None)
        l5_sheets = pd.read_excel(l5_path, sheet_name=None)
        
        # Combine the dictionaries
        combined_sheets = {**f5_sheets, **l5_sheets}
        
        # Write to the output file
        with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
            for sheet_name, df in combined_sheets.items():
                df.to_excel(writer, sheet_name=sheet_name, index=False)
                
        print(f"Successfully merged to: {output_path}")
        return True
    except Exception as e:
        print(f"Error merging {f5_path} and {l5_path}: {str(e)}")
        return False

# Main loop
for target_dir in target_dirs:
    current_path = os.path.join(base_path, target_dir)
    
    if not os.path.exists(current_path):
        print(f"Directory not found: {current_path}")
        continue
        
    print(f"Processing: {current_path}")
    
    # List all directories in the current path
    try:
        subfolders = [f for f in os.listdir(current_path) if os.path.isdir(os.path.join(current_path, f))]
    except Exception as e:
        print(f"Error listing directories in {current_path}: {e}")
        continue

    # Find F5 folders
    f5_folders = [f for f in subfolders if f.startswith('F15S')]
    
    for f5_folder in f5_folders:
        # Extract the suffix (everything after F15S)
        suffix = f5_folder[4:]
        
        # Construct expected L5 and 10S folder names
        l5_folder = f"L15S{suffix}"
        ten_s_folder = f"30S{suffix}"
        
        # Check if L5 and 10S folders exist
        if l5_folder in subfolders and ten_s_folder in subfolders:
            # Construct file paths
            f5_file_name = f"{f5_folder}_training_history.xlsx"
            l5_file_name = f"{l5_folder}_training_history.xlsx"
            ten_s_file_name = f"{ten_s_folder}_training_history.xlsx"
            
            f5_file_path = os.path.join(current_path, f5_folder, f5_file_name)
            l5_file_path = os.path.join(current_path, l5_folder, l5_file_name)
            output_dir_path = os.path.join(current_path, ten_s_folder)
            output_path = os.path.join(current_path, ten_s_folder, ten_s_file_name)
            
            # Check if source files exist
            if os.path.exists(f5_file_path) and os.path.exists(l5_file_path) and os.path.exists(output_dir_path):
                print(f"Merging for {suffix}...")
                merge_histories(f5_file_path, l5_file_path, output_path)
            else:
                if not os.path.exists(f5_file_path):
                    print(f"Missing F5 file: {f5_file_path}")
                if not os.path.exists(l5_file_path):
                    print(f"Missing L5 file: {l5_file_path}")
                if not os.path.exists(output_dir_path):
                    print(f"Missing output directory: {output_dir_path}")

Processing: c:\Files\Degree\FYP\FYP_ver5\Results\30seeds_results
Merging for T2_baseline_algo...
Successfully merged to: c:\Files\Degree\FYP\FYP_ver5\Results\30seeds_results\30ST2_baseline_algo\30ST2_baseline_algo_training_history.xlsx
